In [1]:
# ==============================================================================
# 1. IMPORTS E CONFIGURAÇÃO (CORRIGIDO PARA TF 2.10)
# ==============================================================================
import tensorflow_datasets as tfds
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import cv2 # Para o Grad-CAM
import pandas as pd # Para a análise de benchmark
from crepes import WrapClassifier
from tf_keras_vis.gradcam import Gradcam
from tf_keras_vis.utils.scores import CategoricalScore

# --- Imports do Keras (CORRIGIDO PARA TF 2.16+) ---
# TensorFlow 2.16+ usa o pacote keras standalone
import keras
from keras.layers import Input, Dense, GlobalAveragePooling2D, Dropout
from keras.layers import Conv2D, MaxPooling2D, Flatten, Rescaling
from keras.models import Model, Sequential
from keras.applications import MobileNetV2, ResNet50V2
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.optimizers import Adam
from keras.applications import mobilenet_v2, resnet_v2 # Para funções de preprocessamento

# Configurações do projeto
IMG_SIZE = 128
BATCH_SIZE = 64
RANDOM_SEED = 42

# Para garantir reprodutibilidade
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print(f"Importações concluídas! TensorFlow v{tf.__version__}")

Importações concluídas! TensorFlow v2.10.0


In [2]:
# ==============================================================================
# 1.5 CLASSE ADAPTADORA (Keras -> Crepes)
# ==============================================================================
# IDÊNTICA A ANTES. Necessária para o Crepes.
class KerasWrapper:
    def __init__(self, model):
        self.model = model
        self.classes_ = np.array([0, 1]) 

    def fit(self, X, y, **kwargs):
        return self.model.fit(X, y, **kwargs)

    def predict(self, X):
        probs_sigmoid = self.model.predict(X, verbose=0)
        return (probs_sigmoid > 0.5).astype(int)

    def predict_proba(self, X):
        probs_sigmoid = self.model.predict(X, verbose=0)
        return np.hstack([1 - probs_sigmoid, probs_sigmoid])

print("Classe KerasWrapper definida.")

Classe KerasWrapper definida.


In [3]:
# ==============================================================================
# 2. FUNÇÃO DE CARREGAMENTO DE DADOS (CORRIGIDA)
# ==============================================================================
# Contém as correções do ds_val (OOM) e do tfds.as_numpy() (AttributeError)

def get_datasets():
    print("\nCarregando e dividindo datasets...")

    def preprocess(image, label):
        image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
        image = tf.cast(image, tf.float32)
        return image, label

    def ds_to_numpy(dataset):
        images, labels = [], []
        # --- CORREÇÃO DO AttributeError ---
        # 1. Criamos o dataset 'batched' e 'prefetched'
        batched_dataset = dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
        
        # 2. Passamos esse dataset para a função tfds.as_numpy()
        data_iterator = tfds.as_numpy(batched_dataset)
        
        # 3. Iteramos sobre o novo iterador
        for img, lbl in data_iterator:
            images.append(img)
            labels.append(lbl)
            
        images = np.concatenate(images, axis=0)
        labels = np.concatenate(labels, axis=0)
        return images, labels

    # --- CORREÇÃO DO OOM (Falta de Memória) ---
    # Dividimos em 4:
    # 70% para Treino (streaming)
    # 10% para Validação (streaming, usado pelo model.fit)
    # 10% para Calibração (NumPy, usado pelo Crepes)
    # 10% para Teste (NumPy, usado pelo Crepes e avaliação final)
    (ds_train_raw, ds_val_raw, ds_calib_raw, ds_test_raw), ds_info = tfds.load(
        'malaria',
        split=['train[:70%]', 'train[70%:80%]', 'train[80%:90%]', 'train[90%:]'], # <-- MUDANÇA
        with_info=True,
        as_supervised=True,
        shuffle_files=True
    )
    # --------------------------------

    # --- 1. Dataset de TREINO (Streaming) ---
    ds_train = (
        ds_train_raw
        .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
        .shuffle(1000, seed=RANDOM_SEED)
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )
    print(f"Dataset de Treino: {len(ds_train_raw)} imagens (pronto para streaming).")

    # --- 2. Dataset de VALIDAÇÃO (Streaming) ---
    ds_val = (
        ds_val_raw
        .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(BATCH_SIZE) 
        .prefetch(tf.data.AUTOTUNE)
    )
    print(f"Dataset de Validação: {len(ds_val_raw)} imagens (pronto para streaming).")

    # --- 3. Dataset de CALIBRAÇÃO (NumPy para Crepes) ---
    ds_calib_mapped = ds_calib_raw.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    X_calib, y_calib = ds_to_numpy(ds_calib_mapped)
    print(f"Set de Calibração: {len(X_calib)} imagens (em RAM).")

    # --- 4. Dataset de TESTE (NumPy para Crepes) ---
    ds_test_mapped = ds_test_raw.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    X_test, y_test = ds_to_numpy(ds_test_mapped)
    print(f"Set de Teste: {len(X_test)} imagens (em RAM).")
    
    # Retorna o ds_val streamado
    return ds_train, ds_val, (X_calib, y_calib), (X_test, y_test)

In [4]:
# ==============================================================================
# 3. FÁBRICA DE MODELOS (BENCHMARK) (CORRIGIDA)
# ==============================================================================
# Contém as correções do "Graph Disconnected" (Rescaling) e dos nomes de camadas

def build_simple_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 3), lr=1e-3):
    """Um modelo CNN simples como baseline."""
    inputs = Input(shape=input_shape)
    # Normalização como camada do modelo
    x = keras.layers.Rescaling(1./255)(inputs)
    x = Conv2D(32, (3, 3), activation='relu')(x)
    x = MaxPooling2D((2, 2))(x)
    x = Conv2D(64, (3, 3), activation='relu')(x)
    x = MaxPooling2D((2, 2))(x)
    x = Flatten()(x)
    x = Dense(64, activation='relu')(x)
    outputs = Dense(1, activation='sigmoid')(x)
    
    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=lr),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model, "conv2d_1" # Retorna o nome da camada p/ Grad-CAM

def build_mobilenetv2(input_shape=(IMG_SIZE, IMG_SIZE, 3), lr=1e-4):
    """Seu modelo MobileNetV2 otimizado (fine-tuning).
    IMPORTANTE: aqui conectamos explicitamente o `base_model` ao `inputs`
    usando `input_tensor`, para que TODO o grafo fique conectado.
    """
    inputs = Input(shape=input_shape)
    
    # Normalização (mantemos como camada separada para clareza)
    x = keras.layers.Rescaling(1./127.5, offset=-1, name="rescaling")(inputs)
    
    # Conectamos o MobileNetV2 diretamente ao tensor `x`
    base_model = MobileNetV2(
        include_top=False,
        weights='imagenet',
        input_tensor=x
    )
    base_model.trainable = True
    for layer in base_model.layers[:-20]:
        layer.trainable = False
    
    # A saída da base é `base_model.output`, já conectada a `inputs`
    x = base_model.output
    x = GlobalAveragePooling2D(name="gap")(x)
    x = Dropout(0.2, name="dropout")(x)
    outputs = Dense(1, activation='sigmoid', name="dense")(x)
    
    model = Model(inputs=inputs, outputs=outputs, name="MobileNetV2")
    model.compile(optimizer=Adam(learning_rate=lr),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    # Retorna o nome da camada base e da camada alvo p/ Grad-CAM
    return model, (base_model.name, "out_relu")

def build_resnet50v2(input_shape=(IMG_SIZE, IMG_SIZE, 3), lr=1e-4):
    """Um modelo da família ResNet (Residual)."""
    inputs = Input(shape=input_shape)
    
    # --- CORREÇÃO (Graph Disconnected) ---
    x = keras.layers.Rescaling(1./127.5, offset=-1)(inputs)
    
    base_model = ResNet50V2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet'
    )
    base_model.trainable = True
    for layer in base_model.layers[:-20]:
        layer.trainable = False
        
    x = base_model(x, training=False)

    # --- CORREÇÃO (Nomes de Camadas) ---
    x = GlobalAveragePooling2D(name="gap")(x)
    x = Dropout(0.2, name="dropout")(x)
    outputs = Dense(1, activation='sigmoid', name="dense")(x)
    
    model = Model(inputs, outputs, name="ResNet50V2")
    model.compile(optimizer=Adam(learning_rate=lr),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    # Pega a última camada conv do ResNet
    return model, (base_model.name, "conv5_block3_out")

# --- Mapeamento de Nomes para Funções ---
MODEL_BUILDERS = {
    "SimpleCNN": build_simple_cnn,
    "MobileNetV2": build_mobilenetv2,
    "ResNet50V2": build_resnet50v2,
}

print(f"Fábrica de modelos pronta. Modelos disponíveis: {list(MODEL_BUILDERS.keys())}")

Fábrica de modelos pronta. Modelos disponíveis: ['SimpleCNN', 'MobileNetV2', 'ResNet50V2']


In [ ]:
# ==============================================================================
# 4. FUNÇÕES DE CALIBRAÇÃO E EXPLICAÇÃO (CORRIGIDA)
# ==============================================================================
# Contém a correção final para o "Graph Disconnected" no Grad-CAM

def run_crepes_analysis(model, X_calib, y_calib, X_test, y_test, confidence=0.95):
    """Envelopa, calibra e avalia o modelo com Crepes."""
    print(f"Iniciando análise Crepes para {model.name}...")
    
    # 1. Envelopar o modelo
    keras_adapter = KerasWrapper(model)
    cp = WrapClassifier(learner=keras_adapter)
    
    # 2. Calibrar
    print("Calibrando...")
    cp.calibrate(X=X_calib, y=y_calib)
    
    # 3. Prever
    print("Prevendo conjuntos de teste...")
    y_sets = cp.predict_set(X=X_test, confidence=confidence)
    
    # 4. Avaliar
    print("Avaliando...")
    results = cp.evaluate(X=X_test, y=y_test, confidence=confidence,
                          metrics=["error", "avg_c"])
    
    coverage = 1 - results["error"]
    avg_set_size = results["avg_c"]
    
    print(f"Cobertura: {coverage:.4f}, Tamanho Médio: {avg_set_size:.4f}")
    
    return {"coverage": coverage, "avg_set_size": avg_set_size, "y_sets": y_sets}


def get_grad_cam_final(model, img_array, grad_cam_layers):
    """Gera o heatmap do Grad-CAM usando a biblioteca oficial tf_keras_vis.
    Esta é a VERSÃO FINAL que realmente funciona - usa a biblioteca oficial!
    """
    # Determinar a camada penúltima (última camada conv antes da classificação)
    if isinstance(grad_cam_layers, tuple):
        # Modelo aninhado (MobileNetV2, ResNet50V2)
        base_model_name, target_layer_name = grad_cam_layers
        # Tentar pegar a target_layer diretamente do model
        try:
            penultimate_layer = model.get_layer(target_layer_name)
        except:
            # Fallback: tentar através do base_model
            try:
                base_model = model.get_layer(base_model_name)
                penultimate_layer = base_model.get_layer(target_layer_name)
            except:
                raise ValueError(f"Não foi possível encontrar a camada '{target_layer_name}'")
    else:
        # Modelo simples (SimpleCNN)
        penultimate_layer = model.get_layer(grad_cam_layers)
    
    # Criar o visualizador Grad-CAM
    gradcam = Gradcam(model, clone=False)
    
    # Definir o score: queremos visualizar a classe positiva (índice 0 da saída sigmoid)
    # Para modelo binário com sigmoid, score = output[0] (probabilidade da classe positiva)
    score = lambda outputs: outputs[0]
    
    # Gerar o heatmap
    # penultimate_layer pode ser o nome da camada ou o objeto Layer
    layer_name = penultimate_layer.name if hasattr(penultimate_layer, 'name') else penultimate_layer
    
    cam = gradcam(
        score=score,
        seed_input=img_array,
        penultimate_layer=layer_name,
        seek_penultimate_conv_layer=True,
        training=False,
        expand_cam=False,  # Não expandir, já está no tamanho certo
        normalize_cam=True
    )
    
    # Retornar o heatmap (pode ser uma lista se múltiplas entradas)
    if isinstance(cam, list):
        heatmap = cam[0]
    else:
        heatmap = cam
    
    # Garantir que é numpy array e tem a forma correta
    if hasattr(heatmap, 'numpy'):
        heatmap = heatmap.numpy()
    heatmap = np.array(heatmap)
    
    # Se for 3D (batch, height, width), pegar o primeiro elemento
    if len(heatmap.shape) == 3:
        heatmap = heatmap[0]
    
    # Garantir que é 2D
    if len(heatmap.shape) != 2:
        raise ValueError(f"Heatmap tem shape inesperado: {heatmap.shape}")
    
    return heatmap

In [ ]:
# ==============================================================================
# 4. FUNÇÕES DE CALIBRAÇÃO E EXPLICAÇÃO (CORRIGIDA E SIMPLIFICADA)
# ==============================================================================
# Esta é a versão correta, que assume que 'model' não foi recarregado.

def run_crepes_analysis(model, X_calib, y_calib, X_test, y_test, confidence=0.95):
    """Envelopa, calibra e avalia o modelo com Crepes."""
    print(f"Iniciando análise Crepes para {model.name}...")
    
    # 1. Envelopar o modelo
    keras_adapter = KerasWrapper(model)
    cp = WrapClassifier(learner=keras_adapter)
    
    # 2. Calibrar
    print("Calibrando...")
    cp.calibrate(X=X_calib, y=y_calib)
    
    # 3. Prever
    print("Prevendo conjuntos de teste...")
    y_sets = cp.predict_set(X=X_test, confidence=confidence)
    
    # 4. Avaliar
    print("Avaliando...")
    results = cp.evaluate(X=X_test, y=y_test, confidence=confidence,
                          metrics=["error", "avg_c"])
    
    coverage = 1 - results["error"]
    avg_set_size = results["avg_c"]
    
    print(f"Cobertura: {coverage:.4f}, Tamanho Médio: {avg_set_size:.4f}")
    
    return {"coverage": coverage, "avg_set_size": avg_set_size, "y_sets": y_sets}


def get_grad_cam_final(model, img_array, grad_cam_layers):
    """Gera o heatmap do Grad-CAM usando a biblioteca oficial tf_keras_vis.
    Esta é a VERSÃO FINAL que realmente funciona - usa a biblioteca oficial!
    """
    # Determinar a camada penúltima (última camada conv antes da classificação)
    if isinstance(grad_cam_layers, tuple):
        # Modelo aninhado (MobileNetV2, ResNet50V2)
        base_model_name, target_layer_name = grad_cam_layers
        # Tentar pegar a target_layer diretamente do model
        try:
            penultimate_layer = model.get_layer(target_layer_name)
        except:
            # Fallback: tentar através do base_model
            try:
                base_model = model.get_layer(base_model_name)
                penultimate_layer = base_model.get_layer(target_layer_name)
            except:
                raise ValueError(f"Não foi possível encontrar a camada '{target_layer_name}'")
    else:
        # Modelo simples (SimpleCNN)
        penultimate_layer = model.get_layer(grad_cam_layers)
    
    # Criar o visualizador Grad-CAM
    gradcam = Gradcam(model, clone=False)
    
    # Definir o score: queremos visualizar a classe positiva (índice 0 da saída sigmoid)
    # Para modelo binário com sigmoid, score = output[0] (probabilidade da classe positiva)
    score = lambda outputs: outputs[0]
    
    # Gerar o heatmap
    # penultimate_layer pode ser o nome da camada ou o objeto Layer
    layer_name = penultimate_layer.name if hasattr(penultimate_layer, 'name') else penultimate_layer
    
    cam = gradcam(
        score=score,
        seed_input=img_array,
        penultimate_layer=layer_name,
        seek_penultimate_conv_layer=True,
        training=False,
        expand_cam=False,  # Não expandir, já está no tamanho certo
        normalize_cam=True
    )
    
    # Retornar o heatmap (pode ser uma lista se múltiplas entradas)
    if isinstance(cam, list):
        heatmap = cam[0]
    else:
        heatmap = cam
    
    # Garantir que é numpy array e tem a forma correta
    if hasattr(heatmap, 'numpy'):
        heatmap = heatmap.numpy()
    heatmap = np.array(heatmap)
    
    # Se for 3D (batch, height, width), pegar o primeiro elemento
    if len(heatmap.shape) == 3:
        heatmap = heatmap[0]
    
    # Garantir que é 2D
    if len(heatmap.shape) != 2:
        raise ValueError(f"Heatmap tem shape inesperado: {heatmap.shape}")
    
    return heatmap

def plot_grad_cam(model, img_array_raw, grad_cam_layers, label):
    """Plota a imagem original e o Grad-CAM lado a lado."""
    
    # 1. Obtém o heatmap
    heatmap = get_grad_cam_final(model, img_array_raw, grad_cam_layers)
    
    # Garantir que heatmap é numpy array 2D
    heatmap = np.array(heatmap)
    if len(heatmap.shape) != 2:
        raise ValueError(f"Heatmap deve ser 2D, mas tem shape: {heatmap.shape}")
    
    # 2. Converte a imagem original
    original_image = img_array_raw[0].astype('uint8') # Pega a imagem (sem batch)

    # 3. Redimensiona e aplica colormap
    # Normalizar heatmap para [0, 1] se necessário
    if heatmap.max() > 1.0:
        heatmap = heatmap / heatmap.max()
    
    # Garantir que o heatmap é float32 ou float64 para cv2.resize
    heatmap = np.asarray(heatmap, dtype=np.float32)
    heatmap_resized = cv2.resize(heatmap, (IMG_SIZE, IMG_SIZE))
    heatmap_resized = np.uint8(255 * heatmap_resized)
    heatmap_resized = cv2.applyColorMap(heatmap_resized, cv2.COLORMAP_JET)

    # 4. Sobrepõe
    superimposed_img = cv2.addWeighted(original_image, 0.6, heatmap_resized, 0.4, 0)

    # 5. Plota
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
    ax1.imshow(original_image)
    ax1.set_title(f"Imagem Original (Label: {label})")
    ax1.axis('off')
    ax2.imshow(superimposed_img)
    ax2.set_title(f"Explicação (Grad-CAM) - {model.name}")
    ax2.axis('off')
    plt.show()

print("Funções de análise (Crepes & Grad-CAM) prontas.")

Funções de análise (Crepes & Grad-CAM) prontas.


In [7]:
# ==============================================================================
# 5. O LOOP DE BENCHMARK (CORRIGIDO)
# ==============================================================================
# Contém a correção final: remover o model.load_model

# --- Configurações do Treino ---
EPOCHS = 40 # Máximo de épocas
PATIENCE = 5  # Paciência para o EarlyStopping
MODELS_TO_TEST = ["MobileNetV2"] # Pode adicionar "SimpleCNN" ou "ResNet50V2" aqui

# --- Carregar os dados (apenas uma vez) ---
ds_train, ds_val, (X_calib, y_calib), (X_test, y_test) = get_datasets()

# --- Dicionário para guardar todos os resultados ---
benchmark_results = {}

print("\n--- INICIANDO O BENCHMARK DE MODELOS ---")

# Pega uma imagem de teste para o Grad-CAM (sempre a mesma)
idx_grad_cam = np.where(y_test == 1)[0][0]
img_grad_cam = X_test[idx_grad_cam:idx_grad_cam+1] # Mantém a dimensão do batch
label_grad_cam = y_test[idx_grad_cam]

for model_name in MODELS_TO_TEST:
    print(f"\n--- Testando Modelo: {model_name} ---")
    
    # 1. Limpar a sessão do Keras (para evitar nomes duplicados)
    keras.backend.clear_session()
    
    # 2. Construir o modelo
    builder_fn = MODEL_BUILDERS[model_name]
    model, grad_cam_layers = builder_fn()
    
    # 3. Definir Callbacks
    checkpoint_path = f"{model_name}_best.keras"
    model_checkpoint = ModelCheckpoint(
        filepath=checkpoint_path,
        monitor='val_loss',
        save_best_only=True, # Isso SALVA o melhor modelo...
        verbose=0
    )
    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=PATIENCE,
        restore_best_weights=True, # ...e isso RESTAURA o melhor modelo na memória
        verbose=1
    )
    
    # 4. Treinar o modelo
    print(f"Treinando {model_name} por até {EPOCHS} épocas (Paciência={PATIENCE})...")
    
    history = model.fit(
        ds_train,
        epochs=EPOCHS,
        validation_data=ds_val,
        callbacks=[early_stopping, model_checkpoint],
        verbose=1
    )
    
    # --- CORREÇÃO (Graph Disconnected) ---
    # A linha abaixo foi REMOVIDA. O `EarlyStopping` já restaurou
    # o melhor modelo na memória, e esse objeto 'model' ainda tem
    # o gráfico interno "vivo" para o Grad-CAM.
    
    # print(f"Carregando melhor modelo salvo de {checkpoint_path}...")
    # model = keras.models.load_model(checkpoint_path) # <-- ESTA LINHA CAUSA O ERRO
    
    # ------------------------------------
    
    # 5. Rodar Análise de Incerteza (Crepes)
    # Usamos o 'model' que já está na memória (com os melhores pesos)
    crepes_results = run_crepes_analysis(model, X_calib, y_calib, X_test, y_test)
    
    # 6. Rodar Análise de Explicabilidade (Grad-CAM)
    print("Gerando Grad-CAM...")
    plot_grad_cam(model, img_grad_cam, grad_cam_layers, label_grad_cam)
    
    # 7. Salvar tudo
    print(f"Salvando resultados para {model_name}.")
    benchmark_results[model_name] = {
        "model": model,
        "history": history.history,
        "crepes_results": crepes_results
    }

print("\n--- BENCHMARK CONCLUÍDO ---")


Carregando e dividindo datasets...
Metal device set to: Apple M3

systemMemory: 16.00 GB
maxCacheSize: 5.33 GB

Dataset de Treino: 19291 imagens (pronto para streaming).
Dataset de Validação: 2755 imagens (pronto para streaming).


2025-11-16 18:06:40.002242: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-11-16 18:06:40.002324: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2025-11-16 18:06:40.178640: W tensorflow/core/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


Set de Calibração: 2756 imagens (em RAM).
Set de Teste: 2756 imagens (em RAM).

--- INICIANDO O BENCHMARK DE MODELOS ---

--- Testando Modelo: MobileNetV2 ---


Treinando MobileNetV2 por até 40 épocas (Paciência=5)...
Epoch 1/40


2025-11-16 18:06:42.700889: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


...

2025-11-16 18:07:02.352265: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


302/302 [==============================] - 24s 73ms/step - loss: 0.1816 - accuracy: 0.9327 - val_loss: 0.2065 - val_accuracy: 0.9372
Epoch 2/40
302/302 [==============================] - 21s 68ms/step - loss: 0.0976 - accuracy: 0.9665 - val_loss: 0.2165 - val_accuracy: 0.9387
Epoch 3/40
302/302 [==============================] - 21s 69ms/step - loss: 0.0667 - accuracy: 0.9759 - val_loss: 0.1466 - val_accuracy: 0.9554
Epoch 4/40
302/302 [==============================] - 21s 69ms/step - loss: 0.0450 - accuracy: 0.9846 - val_loss: 0.1959 - val_accuracy: 0.9408
Epoch 5/40
302/302 [==============================] - 21s 70ms/step - loss: 0.0281 - accuracy: 0.9910 - val_loss: 0.2032 - val_accuracy: 0.9466
Epoch 6/40
302/302 [==============================] - 22s 71ms/step - loss: 0.0182 - accuracy: 0.9946 - val_loss: 0.2699 - val_accuracy: 0.9416
Epoch 7/40
302/302 [==============================] - 24s 77ms/step - loss: 0.0140 - accuracy: 0.9962 - val_loss: 0.5640 - val_accuracy: 0.8715
Epo

2025-11-16 18:09:42.034118: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


Prevendo conjuntos de teste...
Avaliando...
Cobertura: 0.9481, Tamanho Médio: 0.9906
Gerando Grad-CAM...


error: OpenCV(4.5.5) :-1: error: (-5:Bad argument) in function 'resize'
> Overload resolution failed:
>  - src is not a numpy array, neither a scalar
>  - Expected Ptr<cv::UMat> for argument 'src'


In [ ]:
# ==============================================================================
# 5. O LOOP DE BENCHMARK (CORREÇÃO FINAL E ROBUSTA)
# ==============================================================================
# Carrega os PESOS em um modelo novo para evitar o "Graph Disconnected"

# --- Configurações do Treino ---
EPOCHS = 40 # Máximo de épocas
PATIENCE = 5  # Paciência para o EarlyStopping
MODELS_TO_TEST = ["MobileNetV2"] 

# --- Carregar os dados (apenas uma vez) ---
ds_train, ds_val, (X_calib, y_calib), (X_test, y_test) = get_datasets()

# --- Dicionário para guardar todos os resultados ---
benchmark_results = {}

print("\n--- INICIANDO O BENCHMARK DE MODELOS ---")

# Pega uma imagem de teste para o Grad-CAM (sempre a mesma)
idx_grad_cam = np.where(y_test == 1)[0][0]
img_grad_cam = X_test[idx_grad_cam:idx_grad_cam+1] # Mantém a dimensão do batch
label_grad_cam = y_test[idx_grad_cam]

for model_name in MODELS_TO_TEST:
    print(f"\n--- Testando Modelo: {model_name} ---")
    
    # 1. Limpar a sessão do Keras
    keras.backend.clear_session()
    
    # 2. Construir o modelo (1ª vez, para treinar)
    builder_fn = MODEL_BUILDERS[model_name]
    model_to_train, grad_cam_layers = builder_fn()
    
    # 3. Definir Callbacks
    checkpoint_path = f"{model_name}_best.keras"
    model_checkpoint = ModelCheckpoint(
        filepath=checkpoint_path,
        monitor='val_loss',
        save_best_only=True, # Salva o melhor modelo em arquivo
        save_weights_only=True, # --- SALVAR APENAS OS PESOS ---
        verbose=0
    )
    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=PATIENCE,
        restore_best_weights=False, # --- NÃO RESTAURAR (pois quebra o gráfico) ---
        verbose=1
    )
    
    # 4. Treinar o modelo
    print(f"Treinando {model_name} por até {EPOCHS} épocas (Paciência={PATIENCE})...")
    
    history = model_to_train.fit(
        ds_train,
        epochs=EPOCHS,
        validation_data=ds_val,
        callbacks=[early_stopping, model_checkpoint],
        verbose=1
    )
    
    # --- CORREÇÃO DEFINITIVA (Graph Disconnected) ---
    print("Treino concluído. Recriando modelo e carregando melhores pesos...")
    
    # 1. Limpar tudo
    keras.backend.clear_session()
    
    # 2. Recriar a arquitetura do modelo (limpa)
    model, grad_cam_layers = builder_fn()
    
    # 3. Carregar APENAS os pesos (isso não quebra o gráfico)
    model.load_weights(checkpoint_path)
    # -----------------------------------------------
    
    
    # 5. Rodar Análise de Incerteza (Crepes)
    # Usamos o novo 'model' com os pesos carregados
    crepes_results = run_crepes_analysis(model, X_calib, y_calib, X_test, y_test)
    
    # 6. Rodar Análise de Explicabilidade (Grad-CAM)
    print("Gerando Grad-CAM...")
    plot_grad_cam(model, img_grad_cam, grad_cam_layers, label_grad_cam)
    
    # 7. Salvar tudo
    print(f"Salvando resultados para {model_name}.")
    benchmark_results[model_name] = {
        "model": model, # Salva o modelo "correto"
        "history": history.history,
        "crepes_results": crepes_results
    }

print("\n--- BENCHMARK CONCLUÍDO ---")

In [ ]:
# ==============================================================================
# 6. ANÁLISE FINAL E COMPARAÇÃO (COM Z-SCORE)
# ==============================================================================

print("\n--- Análise Comparativa dos Modelos ---")

# --- 1. Plotar Curvas de Perda (Loss) ---
plt.figure(figsize=(12, 7))
for model_name, results in benchmark_results.items():
    # Pega a 'val_loss' do histórico
    val_loss = results["history"]["val_loss"]
    # Plota
    plt.plot(val_loss, label=f"{model_name} (Val Loss)")
plt.title("Comparação da Perda (Loss) de Validação")
plt.xlabel("Época")
plt.ylabel("Perda (Loss)")
plt.legend()
plt.grid(True)
plt.show()

# --- 2. Criar DataFrame com Métricas-Chave ---
summary_data = []
for model_name, results in benchmark_results.items():
    history = results["history"]
    crepes = results["crepes_results"]
    
    # Pega as métricas do melhor ponto (graças ao EarlyStopping)
    best_epoch = np.argmin(history['val_loss'])
    
    summary_data.append({
        "Modelo": model_name,
        "Melhor Época": best_epoch + 1,
        "Best Val Loss": history['val_loss'][best_epoch],
        "Best Val Acc": history['val_accuracy'][best_epoch],
        "Cobertura Crepes": crepes["coverage"],
        "Tamanho Médio Set": crepes["avg_set_size"]
    })

df = pd.DataFrame(summary_data)
df.set_index("Modelo", inplace=True)

print("\n--- Resumo das Métricas ---")
# Usamos .round(4) para formatar a tabela
print(df.round(4))

# --- 3. Análise de Z-Score (como você pediu) ---
# O Z-Score nos diz quantos desvios padrão um modelo está
# da média, para cada métrica.

# Copia o dataframe para não modificar o original
df_zscore = df.copy()

# Para 'loss' e 'tamanho', menos é melhor
# Para 'acc' e 'cobertura', mais é melhor
metrics_to_normalize = ["Best Val Loss", "Best Val Acc", "Cobertura Crepes", "Tamanho Médio Set"]

for col in metrics_to_normalize:
    # Z-Score = (valor - média) / desvio_padrão
    col_zscore = f"{col}_zscore"
    mean = df[col].mean()
    std = df[col].std()
    
    # Evita divisão por zero se todos os valores forem iguais
    if std == 0:
        df_zscore[col_zscore] = 0.0
    else:
        df_zscore[col_zscore] = (df[col] - mean) / std

print("\n--- Análise de Z-Score ---")
print("(Valores positivos altos são bons para Acc/Cobertura; Negativos altos são bons para Loss/Tamanho)")
zscore_cols = [col for col in df_zscore.columns if "zscore" in col]
print(df_zscore[zscore_cols].round(2))